# Bloc 2.4 — Spark-style distributed thinking with pandas fallback

**Decision problem:** how would this pipeline change if one machine were not enough?

Output: partition summary using pandas as the local execution engine.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
SNAPSHOT = "iot_FR_today_5-y_chatgpt_iphone_meteo.csv"
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "snapshots" / SNAPSHOT).exists():
        ROOT = candidate
        break
OUT = ROOT / "outputs"
for d in [OUT / "bloc1", OUT / "bloc2", OUT / "bloc3", OUT / "final_product"]: d.mkdir(parents=True, exist_ok=True)
DATA = ROOT / "data" / "snapshots"
def load_clean_long():
    p=OUT/"bloc1"/"clean_trends_long.csv"
    if p.exists(): return pd.read_csv(p, parse_dates=["date"])
    raw=pd.read_csv(DATA/"iot_FR_today_5-y_chatgpt_iphone_meteo.csv", parse_dates=["date"])
    return raw.melt(id_vars="date", var_name="signal", value_name="interest")
def load_clean_wide():
    p=OUT/"bloc1"/"clean_trends_wide.csv"
    if p.exists(): return pd.read_csv(p, parse_dates=["date"])
    raw=pd.read_csv(DATA/"iot_FR_today_5-y_chatgpt_iphone_meteo.csv", parse_dates=["date"])
    return raw.sort_values("date")

In [ ]:
df=load_clean_long()
df["year"]=df["date"].dt.year
partition_summary=df.groupby(["year","signal"]).agg(rows=("interest","size"), avg_interest=("interest","mean"), max_interest=("interest","max")).reset_index().round(2)
partition_summary.to_csv(OUT/"bloc2"/"partition_summary.csv", index=False)
partition_summary.head(12)

## Exercise

Identify which operation is map-like and which is reduce-like.

## Conclusion

Distributed thinking is about partitioning, aggregating, and minimizing data movement; pandas is enough to teach the pattern.